In [1]:
!pip install -q transformers accelerate sentencepiece

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [4]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/DialoGPT-medium"
).to(device)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [5]:
chat_history_ids = None

print("Chatbot is ready!")
print("Type 'quit' to exit.")

Chatbot is ready!
Type 'quit' to exit.


In [6]:
while True:

    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Chat ended.")
        break

    # Encode the user input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # Build conversation history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Keep only the most recent tokens if history becomes too long
    if bot_input_ids.shape[-1] > 800:
        bot_input_ids = bot_input_ids[:, -800:]

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=bot_input_ids.shape[-1] + 60,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.9
    )

    # Decode only the newly generated response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    if response == "":
        response = "Sorry, I couldn't think of a response."

    print("Bot:", response)

You: Hi


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Bot: Hello , how are you ?
You: I am good. How are you?
Bot: I'm good thanks .
You: What's your name?
Bot: My name is Alistair .
You: What can you do?
Bot: I can make people laugh .
You: Tell me about AI.
Bot: Can I ?
You: yes you can
Bot: I can
You: ok bye
Bot: See ya later .
You: quit
Chat ended.
